# 🎬 Session 1: Foundations and a TF-IDF recommender (solution)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pearlin-in/recommendationsystem---GDGHWU/blob/main/notebooks/session1_solution.ipynb)

**GDG on Campus · Build Your Own Recommendation System**

This is the **solution notebook**: every ✏️ TODO is already filled in. Use it to catch up if you fall behind, or to compare with your own answers.
The version with blanks to fill in is [`session1_starter.ipynb`](https://colab.research.google.com/github/pearlin-in/recommendationsystem---GDGHWU/blob/main/notebooks/session1_starter.ipynb).

| Part | What you will do | ⏱ |
|---|---|---|
| 0 | Set up Google Colab | 5 min |
| 1 | What is a recommender system? | 5 min |
| 2 | Load 4,800 movies | 5 min |
| 3 | Pandas in 10 minutes | 15 min |
| 4 | Clean the data and build a text "soup" | 15 min |
| 5 | Vectors and cosine similarity | 15 min |
| 6 | TF-IDF: turning text into vectors | 15 min |
| 7 | Build `recommend()` | 15 min |
| 8 | Try your favourite movies | 10 min |

**Deliverable:** `recommend("The Dark Knight")` returns 5 sensible movies.

---
## Part 0 · Setup

Google Colab gives you a free Python machine in the cloud. The default **CPU** runtime is all we need today: no GPU, no installs.
Everything we use (pandas, NumPy, scikit-learn, matplotlib and Hugging Face `datasets`) is already installed on Colab.

In [ ]:
import sys
print("Python", sys.version.split()[0])

try:
    import google.colab  # only exists inside Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Running in Google Colab ✅" if IN_COLAB else "Running locally (that works too)")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

# This workshop's GitHub repo. It holds backup copies of the data in case a download fails.
REPO_RAW = "https://raw.githubusercontent.com/pearlin-in/recommendationsystem---GDGHWU/main/"


def check(test, hint):
    """Tiny auto-grader used by the 🧪 Check cells."""
    try:
        assert test()
        print("✅ Nice, that works!")
    except Exception:
        print("❌ Not quite yet. Hint:", hint)

---
## Part 1 · What is a recommender system?

There are two big families:

| | **Content-based** (what we build) | **Collaborative filtering** |
|---|---|---|
| Looks at | what an item *is*: plot, genres, keywords | what *people did*: ratings, clicks, plays |
| Sounds like | "This is another dark superhero crime film" | "People who liked X also liked Y" |
| Needs user data? | No | Yes, lots of it |
| Brand-new item | Works straight away | Can't recommend it until people use it ("cold start") |

Netflix, Spotify, YouTube and Google Play mix both. Today we build the content-based kind:

```
 "Inception"  ->  [ turn every movie into a vector ]  ->  [ compare with every other vector ]  ->  top 5 closest
```

**Why this is unsupervised learning:** nobody tells the model what the "right" recommendation is. There are no labels.
It discovers which movies are close together from the data alone.

---
## Part 2 · Load the movies

We use **TMDB 5000**: about 4,800 movies with plot summaries, genres and keywords. It downloads from Hugging Face with one line, with no login and no API key.

In [ ]:
from datasets import load_dataset

try:
    raw = load_dataset("AiresPucrs/tmdb-5000-movies", split="train").to_pandas()
except Exception as e:  # Hugging Face down or slow? Use the copy in our repo instead.
    print("Hugging Face download failed, using the backup copy:", e)
    raw = pd.read_parquet(REPO_RAW + "data/tmdb_5000_movies.parquet")

print(raw.shape)  # (rows, columns)
raw.head(3)

---
## Part 3 · Pandas in 10 minutes

A pandas **DataFrame** is a table: one row per movie, one column per property. Here are the moves you will use all day.

In [ ]:
# 1. What columns do we have?
raw.columns.tolist()

In [ ]:
# 2. Pick rows with a condition, and columns with a list
raw.loc[raw["title"] == "The Dark Knight", ["title", "release_date", "vote_average", "overview"]]

In [ ]:
# 3. How many values are missing in each column?
raw.isna().sum().sort_values(ascending=False).head(6)

In [ ]:
# 4. Sort: the movies with the most votes
raw.sort_values("vote_count", ascending=False)[["title", "vote_count", "vote_average"]].head(10)

In [ ]:
# 5. Combine conditions with & (and) or | (or). Wrap each condition in brackets.
raw[(raw["vote_average"] >= 8) & (raw["runtime"] < 100)][["title", "vote_average", "runtime"]]

### ✏️ Exercise 1: the best-rated movies

Find the **10 highest-rated movies with at least 1,000 votes**. Show `title`, `vote_average` and `vote_count`.

*Why require 1,000 votes?* A movie with a single 10/10 vote should not beat one rated 8.5 by 8,000 people. Keep this idea in mind: popularity and reliability matter for recommenders too.

In [ ]:
top_rated = (
    raw[raw["vote_count"] >= 1000]
    .sort_values("vote_average", ascending=False)
    [["title", "vote_average", "vote_count"]]
    .head(10)
)
top_rated

In [ ]:
# 🧪 Check
check(lambda: len(top_rated) == 10
              and top_rated["vote_count"].min() >= 1000
              and top_rated["vote_average"].is_monotonic_decreasing,
      "raw[raw['vote_count'] >= 1000], then .sort_values('vote_average', ascending=False), then .head(10)")

---
## Part 4 · Clean the data

Look at the `genres` column. It *looks* like a list, but it is really one long piece of text in JSON format:

In [ ]:
print(raw.loc[0, "genres"])
print(type(raw.loc[0, "genres"]))

`json.loads` turns that text into a real Python list of dictionaries. We only want the `"name"` of each one.

In [ ]:
json.loads(raw.loc[0, "genres"])

### ✏️ Exercise 2: write `names()`

In [ ]:
def names(cell):
    """'[{"id": 28, "name": "Action"}, ...]'  ->  ["Action", ...]"""
    try:
        items = json.loads(cell)
        return [d["name"] for d in items]
    except Exception:  # missing or broken values become an empty list
        return []

In [ ]:
# 🧪 Check
check(lambda: names('[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}]') == ["Action", "Adventure"]
              and names(None) == [],
      "loop over items and collect d['name'], for example [d['name'] for d in items]")

In [ ]:
movies = raw.copy()
movies["overview"] = movies["overview"].fillna("")
movies["genre_list"] = movies["genres"].apply(names)
movies["keyword_list"] = movies["keywords"].apply(names)
movies["year"] = pd.to_datetime(movies["release_date"], errors="coerce").dt.year.fillna(0).astype(int)

movies[["title", "year", "genre_list", "keyword_list"]].head()

In [ ]:
# A quick look at the data: which genres are most common?
genre_counts = movies["genre_list"].explode().value_counts()
genre_counts.head(12).sort_values().plot.barh(figsize=(7, 4), title="Most common genres")
plt.xlabel("number of movies")
plt.show()

### The "soup"

We glue everything we know about a movie into **one piece of text**, the *soup*: overview + genres + keywords.
The model will only ever see this text, so **the soup defines what "similar" means**.

In [ ]:
movies["soup"] = (
    movies["overview"]
    + " Genres: " + movies["genre_list"].apply(", ".join)
    + ". Keywords: " + movies["keyword_list"].apply(", ".join)
)

# Most-voted first (so partial title searches find the famous movie), drop duplicate titles
# and drop movies with almost no text.
movies = movies.sort_values("vote_count", ascending=False, kind="stable").drop_duplicates("title")
movies = movies[movies["soup"].str.len() > 30].reset_index(drop=True)

print(movies.shape)
print(movies.loc[movies["title"] == "The Dark Knight", "soup"].iloc[0])

---
## Part 5 · Vectors and cosine similarity

A **vector** is just a list of numbers. Imagine scoring movies on three made-up features:

In [ ]:
#                  action  romance  space
toy = pd.DataFrame(
    [[9, 1, 8],    # Star Wars
     [8, 2, 9],    # Guardians of the Galaxy
     [1, 9, 0],    # The Notebook
     [2, 8, 1]],   # Pride & Prejudice
    columns=["action", "romance", "space"],
    index=["Star Wars", "Guardians of the Galaxy", "The Notebook", "Pride & Prejudice"],
)
toy

In [ ]:
# Draw each movie as an arrow, using only the action and romance numbers
plt.figure(figsize=(4.5, 4.5))
for name, row in toy.iterrows():
    plt.arrow(0, 0, row["action"], row["romance"], head_width=0.25, length_includes_head=True, color="tab:blue")
    plt.text(row["action"] + 0.2, row["romance"], name)
plt.xlim(0, 13); plt.ylim(0, 10); plt.xlabel("action"); plt.ylabel("romance")
plt.title("Similar movies point in similar directions")
plt.show()

**Cosine similarity** measures the angle between two arrows:

$$\text{cosine}(a, b) = \frac{a \cdot b}{\|a\| \times \|b\|}$$

- $a \cdot b$ is the **dot product**: multiply matching numbers and add them up.
- $\|a\|$ is the **length** of $a$: the square root of the sum of its squares.
- `1.0` means the arrows point the same way (very similar), `0.0` means unrelated.
- It compares **direction, not length**, so a long plot summary and a short one about the same topic still match.

### ✏️ Exercise 3: write `cosine()`

In [ ]:
def cosine(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
# 🧪 Check
check(lambda: abs(cosine([1, 0], [0, 1])) < 1e-9 and abs(cosine([1, 2], [2, 4]) - 1) < 1e-9,
      "np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))")

In [ ]:
print("Star Wars vs Guardians of the Galaxy:", round(cosine(toy.loc["Star Wars"], toy.loc["Guardians of the Galaxy"]), 3))
print("Star Wars vs The Notebook:           ", round(cosine(toy.loc["Star Wars"], toy.loc["The Notebook"]), 3))
print("[1, 1, 0] vs [10, 10, 0] (same direction, 10x longer):", round(cosine([1, 1, 0], [10, 10, 0]), 3))

In [ ]:
# scikit-learn compares every pair at once
pd.DataFrame(cosine_similarity(toy), index=toy.index, columns=toy.index).round(2)

💡 **The big idea:** if we can turn every movie into a vector, then *"recommend similar movies"* becomes *"find the vectors pointing in the closest direction"*.
We can't score 4,800 movies by hand, so next we let TF-IDF build the vectors from the text.

---
## Part 6 · TF-IDF: turning text into vectors

Counting words is a good start, but words like "the", "man" and "life" appear everywhere and say little. **TF-IDF** fixes that:

- **TF (term frequency):** how often a word appears in *this* movie's text.
- **IDF (inverse document frequency):** how *rare* the word is across *all* movies.
- **TF × IDF** is high for words that are frequent here but rare elsewhere ("joker", "gotham") and low for common words.

Every distinct word gets its own column, so each movie becomes a long vector of word scores. A tiny example first:

In [ ]:
docs = [
    "a boy wizard goes to a school of magic",
    "a young wizard fights a dark lord",
    "a shark attacks a quiet beach town",
]
demo = TfidfVectorizer()
D = demo.fit_transform(docs)
pd.DataFrame(D.toarray(), columns=demo.get_feature_names_out(), index=["doc 1", "doc 2", "doc 3"]).round(2)

In [ ]:
# doc 1 and doc 2 share the word "wizard", doc 3 shares nothing with them
pd.DataFrame(cosine_similarity(D), index=["doc 1", "doc 2", "doc 3"], columns=["doc 1", "doc 2", "doc 3"]).round(2)

### ✏️ Exercise 4: TF-IDF for every movie

Create a `TfidfVectorizer` with these settings, then call `.fit_transform()` on the soup column:

| Setting | Why |
|---|---|
| `stop_words="english"` | ignore filler words such as "the", "and", "of" |
| `ngram_range=(1, 2)` | also use word pairs, such as "dark knight" or "time travel" |
| `max_features=20000` | keep only the 20,000 most common terms to save memory |

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=20000)
X = tfidf.fit_transform(movies["soup"])

In [ ]:
# 🧪 Check
check(lambda: X.shape == (len(movies), 20000),
      "TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=20000), then tfidf.fit_transform(movies['soup'])")

In [ ]:
print("Matrix shape (movies, terms):", X.shape)
print(f"Only {X.nnz / (X.shape[0] * X.shape[1]):.2%} of the numbers are non-zero, so it is stored as a *sparse* matrix")

---
## Part 7 · Build `recommend()`

The plan:
1. Find the row of the movie the user typed.
2. Compute the cosine similarity between **that one row** and every row. That gives one score per movie.
   (We never build the full 4,800 × 4,800 table. That works here, but for 100,000 songs it would need about 50 GB.)
3. Make sure the movie itself is never recommended, since it is 100% similar to itself.
4. Sort the scores and return the top `k`.

First, a helper that finds the row of a title (ignoring capital letters), and suggests titles if it can't:

In [ ]:
def find_index(query, df, title_col="title"):
    """Row position of the item whose title matches `query` (case-insensitive)."""
    titles = df[title_col].str.lower()
    matches = np.flatnonzero(titles == query.lower())
    if len(matches) == 0:
        close = df.loc[titles.str.contains(query.lower(), regex=False), title_col].head(5).tolist()
        raise ValueError(f"'{query}' not found. Did you mean: {close}")
    return int(matches[0])


find_index("The Dark Knight", movies)

### ✏️ Exercise 5: the recommender

Note on `vectors[i:i+1]`: it selects row `i` but keeps it 2-D (one row, many columns), which is the shape `cosine_similarity` expects.

In [ ]:
def recommend(query, vectors, df, title_col="title", k=5):
    """Return the k items most similar to `query`.
    `vectors` can be a sparse TF-IDF matrix or a dense NumPy array, with one row per item in `df`."""
    i = find_index(query, df, title_col)
    sims = cosine_similarity(vectors[i:i+1], vectors).ravel()  # one row vs all rows, never the full matrix
    sims[i] = -np.inf                                          # never recommend the item itself
    top = np.argsort(-sims)[:k]                                # positions of the k highest scores
    out = df.iloc[top].copy()
    out["similarity"] = sims[top].round(3)
    return out

> **Why not just skip the first result?** If two movies had identical text, both would score 1.0, and the movie itself might not come first.
> Setting its own score to `-inf` is always safe.

In [ ]:
recs = recommend("The Dark Knight", X, movies)
recs[["title", "year", "similarity"]]

In [ ]:
# 🧪 Check: the deliverable for Session 1
check(lambda: len(recs) == 5
              and "The Dark Knight Rises" in recs["title"].values
              and "The Dark Knight" not in recs["title"].values,
      "re-read the three TODOs inside recommend()")

🎉 **That's a working recommender**, built with no labels and no training data. Pure unsupervised learning.

---
## Part 8 · Explore it, break it, improve it

In [ ]:
for title in ["Toy Story", "The Matrix", "Titanic", "Up"]:
    print(f"\nBecause you liked {title}:")
    print(recommend(title, X, movies)[["title", "year", "similarity"]].to_string(index=False))

### ✏️ Exercise 6: your favourites

Try at least 3 movies you love. If a title is not found, the error message suggests the exact spelling.

In [ ]:
recommend("Interstellar", X, movies)[["title", "year", "genre_list", "similarity"]]

**TF-IDF matches words, not meaning.** Two plots about the same idea told with different words look unrelated to it.
Session 2 fixes this with **embeddings**, which match meaning.

---
## ✅ Recap

| You learned | In one line |
|---|---|
| Content-based recommenders | Recommend items whose *content* is similar |
| Unsupervised learning | No labels: structure comes from the data alone |
| Vectors and cosine similarity | Similar items point in similar directions |
| TF-IDF | Turns text into vectors that weight rare, telling words highly |
| `recommend()` | One row vs all rows, drop itself, sort, take top k |

**Next session:** sentence embeddings (meaning instead of words), comparing the two approaches, Gemini explanations and a shareable web app.